In [2]:
# Writing .backward() ourself, instead of using Pytorchs
# for better understanding, optimization and debuging

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
words = open("names.txt", 'r').read().splitlines()

In [4]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i : s for s, i in stoi.items()}
print(itos)

vocab_size = len(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# building the dataset
block_size = 3 # context length

def build_dataset(words):
    
    X, Y = [], []
    for w in words:
    
        # print(w)
        context = [0] * block_size
        for ch in w +'.' :
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            # print(''.join(itos[i] for i in context), '--->', itos[ix])
            context = context[1:] + [ix] # crop and append
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
# utility function we will use later when comparing manual gradients to Pytorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [7]:
n_embd = 10
n_hidden = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size , n_embd),    generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn((n_hidden),                      generator=g) * 0.1 # it's useless as we are using Batch norm
#Layer 2
W2 =  torch.randn((n_hidden, vocab_size),         generator=g) * 0.1
b2 = torch.randn((vocab_size),                    generator=g) * 0.1 
# Batchnorm Parameters
bngain = torch.randn((1, n_hidden)) * 0.1 + 1
bnbias = torch.randn((1, n_hidden)) * 0.1

# Note: initialized non-standard way ( not with ones or zeros ) as init with eg. all zeros could mask an incorrect
# implementation of the backward pass

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

4137


In [8]:
batch_size = 32
n = batch_size # a shorter evar also, for convenience
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y

In [42]:
# Forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb]  # embed character into vectors
embcat = emb.view(emb.shape[0], -1) # concat the vectors
# Linear Layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# Bachnorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff ** 2
bnvar = 1/(n-1) * (bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not 1)
bnvar_inv = (bnvar +1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# non linearity
h = torch.tanh(hpreact) # hidden layer
# Linear Layer 3
logits = h @ W2 + b2 # output layer
# corss_entropy loss
logit_maxes = logits.max(1, keepdim=True).values
norm_logits  = logits - logit_maxes # substract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1 # if we use 1/count_sum instead, can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# pytorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, bnmeani, hprebn,
          embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.3470, grad_fn=<NegBackward0>)

In [76]:
embcat.shape, W1.shape, C.shape, demb.shape, Xb.shape

(torch.Size([32, 30]),
 torch.Size([30, 64]),
 torch.Size([27, 10]),
 torch.Size([32, 3, 10]),
 torch.Size([32, 3]))

In [44]:
# we check what is the effect of current term on what going to happen next i.e. the output of logprobs
# on loss

# dlogporbs = dloss/dia
# loss = -(1/n * i1 + ..... 1/n * in)   : -ve mean
# dloss/dia = -1/n

# dprobs = dlogprobs/dx
# dlogprob/dx = 1/x  ( x belongs to logprobs )

# dcounts_sum_inv = dprobs/dx
# 2 operation - broadcasting (same value for whole row) -> multiplication ( 2 scalars )
# dprobs/dx = (counts[x]). sum[row] (as across diff. branches using same x, gradients are added)

# dcounts_sum = dcounts_sum_inv/dcounts_sum
# dcounts_sum_inv/counts_sum = -1 * (counts_sum**-2)

# dcounts = dprobs/dcounts + dcounts_sum/dcounts 
# for probs = counts * counts_sum_inv
# dprobs/dcounts = counts_sum_inv
# for counts_sum = counts.sum(1, keepdim=True)
# dcounts_sum/dcounts[x] = dcounts_sum[x] ( as it's basic addition of row wise element the gradient same gradient flows on)

# dnorm_logits = dcounts/dx
# which is exp of norm_logits, which is basically exp (same)
# dcounts/dx = norm_logits[x].exp()

# dlogit_maxes = dnorm_logits/dx
# 2 operation - broadcasting (same value for whole row) -> substraction ( 2 scalars )
# dnorm_logits/dx = (-1 * incomming_grad). sum[row] (as across diff. branches using same x, gradients are added)

# dlogits = dnorm_logits/dx + dlogit_maxes/dx
# for norm_logits  = logits - logit_maxes
# dnorm_logits/dx = 1
# for logit_maxes = logits.max(1, keepdim=True).values
# dlogit_maxes/dx = 1 for the max position in row else 0

# dh = dlogits/dx
# dlogits/dx = dlogits @ W2(transpose)

# Similarly,
# dW2 = h(transpose) @ dlogits

# db2 = 1
# then we sum over columns as it is broadcasted on 0

# dhpreact = dh/dhpreact
# dh/dhpreact = 1 - tanh(hpreact)**2 = 1 - h**2

# for dbngain and dbnbias, simple multiply and addion with broadcasting at 0

# for bnraw, as there is scalar mult -> we get the 2nd operand (bngain)

# dbnvar_inv = simple mult, broadcasted at 0

# dbnvar = power rule with simple addition

# dbndiff2 = summed at 0 (ones of size bndiff2) and then mult

# dbndiff = dbnraw/dx + dbndiff2/dx
# dbnraw/dx = simple mult
# dbndiff2/dx = power rule 

# dbnmeani = simple subs, then broadcasted at 0

# dhprebn = dbndiff/dx + dbnmeani/dx
# dbndiff/dx = simple sub (first term) 
# dbnmeani/dx = summed at 0 (ones of size bndiff2) and then mult

# for dW1, dembcatn -> matrix mult
# db1 = simple addition, then broadcast at 0

# demb = not change in val, so 1 (just change in view of emb)

# dC = unwrap the indexing of emb, and if rows of C were used multiple times we add the gradients

In [75]:
# Excercise 1: backrop through the whole thing manually 1-by-1

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/logprobs.shape[0]
dprobs = (1.0/probs) * dlogprobs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts_sum = -1 * (counts_sum**-2) * dcounts_sum_inv
dcounts = (counts_sum_inv * dprobs) + torch.ones_like(counts) * dcounts_sum
dnorm_logits = norm_logits.exp() * dcounts
dlogit_maxes = -dnorm_logits.sum(1, keepdim=True)
dlogits = dnorm_logits.clone() + F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
dhpreact = (1 - h**2) * dh
dbngain = (bnraw * dhpreact).sum(0)
dbnbias = dhpreact.sum(0)
dbnraw = bngain * dhpreact
dbnvar_inv = (bndiff * dbnraw).sum(0)
dbnvar = -0.5*((bnvar+1e-5)**-1.5) * dbnvar_inv
dbndiff2 = 1.0/(n-1) * torch.ones_like(bndiff2) * dbnvar
dbndiff = bnvar_inv * dbnraw + 2 * bndiff * dbndiff2
dbnmeani = -dbndiff.sum(0)
dhprebn = dbndiff + torch.ones_like(hprebn)*1.0/n * dbnmeani
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape[0], block_size, -1)
dC = torch.zeros_like(C)
for i in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        dC[Xb[i, j]] += demb[i, j]

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff:

In [57]:
h.shape, W2.shape, dlogits.shape, b2.shape

(torch.Size([32, 64]),
 torch.Size([64, 27]),
 torch.Size([32, 27]),
 torch.Size([27]))

In [77]:
# Exercise 2: backprop through cross_entropy but all in one go


# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.3470282554626465 diff: -2.384185791015625e-07


In [80]:
# backward pass

dlogits = F.softmax(logits, 1) # along the rows
dlogits[range(n), Yb] -= 1 # we subs 1 from the true label logit
dlogits /= n # to diffrentiate the mean

cmp('logits', dlogits, logits) # not exact bit wise due to floating point error

logits          | exact: False | approximate: True  | maxdiff: 5.122274160385132e-09


In [78]:
logits.shape, Yb.shape

(torch.Size([32, 27]), torch.Size([32]))

In [89]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [90]:
# backward pas

dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)* bnraw*(dhpreact*bnraw).sum(0))
# sum at 0 as it's summing the y's not x's

cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


In [91]:
bngain.shape, bnvar_inv.shape, dhpreact.shape, bnraw.shape, dhpreact.sum(0).shape, dhprebn.shape

(torch.Size([1, 64]),
 torch.Size([1, 64]),
 torch.Size([32, 64]),
 torch.Size([32, 64]),
 torch.Size([64]),
 torch.Size([32, 64]))

In [103]:
# Exercise 4: putting it all together!
# Train the MLP neural net with your own backward pass

#init
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size , n_embd),    generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn((n_hidden),                      generator=g) * 0.1 # it's useless as we are using Batch norm
#Layer 2
W2 =  torch.randn((n_hidden, vocab_size),         generator=g) * 0.1
b2 = torch.randn((vocab_size),                    generator=g) * 0.1 
# Batchnorm Parameters
bngain = torch.randn((1, n_hidden)) * 0.1 + 1
bnbias = torch.randn((1, n_hidden)) * 0.1

# Note: initialized non-standard way ( not with ones or zeros ) as init with eg. all zeros could mask an incorrect
# implementation of the backward pass

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size 
lossi = []

# use this context manager for efficiency once your backward pass is written (TODO)
with torch.no_grad():

    for i in range(max_steps):
    
        # mini batch construch 
        ix = torch.randint(0, Xtr.shape[0], (batch_size, ), generator=g)
        Xb, Yb = Xtr[ix], Ytr[ix]
    
        # forward pass
        emb = C[Xb]
        embcat = emb.view(emb.shape[0], -1)
        # Linear layer
        hprebn = embcat @ W1 + b1 # hidden layer pre-activation
        # BatchNorm layer
        # -------------------------------------------------------------
        bnmean = hprebn.mean(0, keepdim=True)
        bnvar = hprebn.var(0, keepdim=True, unbiased=True)
        bnvar_inv = (bnvar + 1e-5)**-0.5
        bnraw = (hprebn - bnmean) * bnvar_inv
        hpreact = bngain * bnraw + bnbias
        # -------------------------------------------------------------
        # Non-linearity
        h = torch.tanh(hpreact) # hidden layer
        logits = h @ W2 + b2 # output layer
        loss = F.cross_entropy(logits, Yb) # loss function
    
        # backward pass
        for p in parameters:
          p.grad = None
        # loss.backward() # use this for correctness comparisons, delete it later!
    
        # manual backprop:
    
        # exercise 2: loss back prop
        dlogits = F.softmax(logits, 1) # along the rows
        dlogits[range(n), Yb] -= 1 # we subs 1 from the true label logit
        dlogits /= n # to diffrentiate the mean
    
        # 2nd layer backprop
        dh = dlogits @ W2.T
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(0)
    
        # tanh backprop
        dhpreact = (1 - h**2) * dh
        
        # exerxise 3: batch norm back prop
        dbngain = (bnraw * dhpreact).sum(0)
        dbnbias = dhpreact.sum(0)
        dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)* bnraw*(dhpreact*bnraw).sum(0))
    
        # 1st layer backprop
        dembcat = dhprebn @ W1.T
        dW1 = embcat.T @ dhprebn
        db1 = dhprebn.sum(0)
    
        # embedding backprop
        demb = dembcat.view(emb.shape[0], block_size, -1)
        dC = torch.zeros_like(C)
        for k in range(Xb.shape[0]):
            for j in range(Xb.shape[1]):
                dC[Xb[k, j]] += demb[k, j]     
        grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
    
        # update
        lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
        for p, grad in zip(parameters, grads):
          #p.data += -lr * p.grad # old way of cheems doge (using PyTorch grad from .backward())
          p.data += -lr * grad # new way of swole doge TODO: enable
    
        # track stats
        if i % 10000 == 0: # print every once in a while
          print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
        lossi.append(loss.log10().item())
    
        # if i >= 100:
        #     break

12297
      0/ 200000: 3.7741
  10000/ 200000: 2.1606
  20000/ 200000: 2.4245
  30000/ 200000: 2.5047
  40000/ 200000: 1.9587
  50000/ 200000: 2.4145
  60000/ 200000: 2.3844
  70000/ 200000: 2.0811
  80000/ 200000: 2.3364
  90000/ 200000: 2.1023
 100000/ 200000: 1.9403
 110000/ 200000: 2.3780
 120000/ 200000: 1.9847
 130000/ 200000: 2.4834
 140000/ 200000: 2.3608
 150000/ 200000: 2.1338
 160000/ 200000: 1.9305
 170000/ 200000: 1.8585
 180000/ 200000: 1.9737
 190000/ 200000: 1.9145


In [102]:
# useful for checking your gradients
for p,g in zip(parameters, grads): # for i >= 100
  cmp(str(tuple(p.shape)), g, p)

(27, 10)        | exact: False | approximate: True  | maxdiff: 1.30385160446167e-08
(30, 200)       | exact: False | approximate: True  | maxdiff: 1.1175870895385742e-08
(200,)          | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09
(200, 27)       | exact: False | approximate: True  | maxdiff: 1.1175870895385742e-08
(27,)           | exact: False | approximate: True  | maxdiff: 7.450580596923828e-09
(1, 200)        | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09
(1, 200)        | exact: False | approximate: True  | maxdiff: 5.587935447692871e-09


In [104]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)

In [105]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0705018043518066
val 2.1118295192718506


In [106]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # ------------
      # forward pass:
      # Embedding
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # ------------
      # Sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

carla.
fati.
harli.
jari.
reity.
skaelane.
mahnen.
delynn.
jareei.
ner.
kia.
chaiha.
kaleigh.
ham.
joce.
quint.
salin.
alianni.
wavero.
dearynix.
